In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
import vectorbt as vbt
from datetime import datetime, timedelta
import warnings
import tqdm as tqdm
from tabulate import tabulate
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.graph_objects as go
from datetime import datetime
import hashlib
import itertools
import os
import json 
import mplfinance as mpf
warnings.filterwarnings('ignore') 

ModuleNotFoundError: No module named 'mplfinance'

In [ ]:
data_path = r"C:\Users\OMEN\Desktop\QUANT INVESTMENT TEAM\turtle trading final\data\nifty500\nifty500_daily_ohlcv.csv"
data = pd.read_csv(data_path, header=[0,1], index_col=0, parse_dates=True)



In [ ]:
def calculate_atr(df, period=14):
    high = df['High']
    low = df['Low']
    close = df['Close']

    prev_close = close.shift(1)

    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low - prev_close).abs()
    ], axis=1).max(axis=1)

    atr = tr.rolling(period).mean()
    
    return atr


In [ ]:
def calculate_donchian(df):
    upper = df['High'].rolling(LOOKBACK).max().shift(1)
    lower = df['Low'].rolling(LOOKBACK).min().shift(1)
    mid = (upper + lower) / 2
    return upper, lower, mid


In [ ]:
# param_grid = {
#     "LOOKBACK": [5, 10, 20, 40, 60, 120, 180, 240],
#     "SL_ATR_MULT": [5, 10, 15, 20, 25, 30, 35, 40],
#     "RR_RATIO": [1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6],
#     "MAX_STOCKS": [5, 7, 10, 12, 15, 20, 25, 30],
#     "ATR_PERIOD": [14],
#     "MOMENTUM": [True, False]
# }


In [ ]:
INITIAL_CAPITAL=1000000

In [ ]:
def precompute_indicators(data: pd.DataFrame) -> dict:
    """
    Precompute ATR and Donchian for all tickers once.
    Returns a dict of DataFrames.
    """

    tickers = data.columns.get_level_values(0).unique()
    index = data.index

    atr_df = pd.DataFrame(index=index, columns=tickers, dtype=float)
    donchian_upper = pd.DataFrame(index=index, columns=tickers, dtype=float)
    donchian_lower = pd.DataFrame(index=index, columns=tickers, dtype=float)

    for ticker in tickers:
        df = data[ticker]

        # ATR (shifted for lookahead safety)
        atr_df[ticker] = calculate_atr(df, ATR_PERIOD).shift(1)

        # Donchian (already shifted internally)
        upper, lower, _ = calculate_donchian(df)
        donchian_upper[ticker] = upper
        donchian_lower[ticker] = lower

    return {
        "ATR": atr_df,
        "DONCHIAN_UPPER": donchian_upper,
        "DONCHIAN_LOWER": donchian_lower,
    }


In [ ]:
class Strategy:
    """
    Donchian Channel strategy (signals only).

    Signal at index t is EXECUTED at OPEN[t]
    Uses information available only up to CLOSE[t-1]
    """

    def __init__(self, indicators: dict):
        self.signals_df = None
        self.indicators = indicators

    def generate_signals_multi_ticker(self, data: pd.DataFrame) -> pd.DataFrame:
        tickers = data.columns.get_level_values(0).unique()
        index = data.index

        signals = pd.DataFrame(
            0,
            index=index,
            columns=tickers,
            dtype=float
        )

        atr_df = self.indicators["ATR"]
        upper_df = self.indicators["DONCHIAN_UPPER"]
        lower_df = self.indicators["DONCHIAN_LOWER"]

        for ticker in tickers:
            df = data[ticker]

            atr = atr_df[ticker]
            upper = upper_df[ticker]
            lower = lower_df[ticker]

            in_position = False
            entry_price = None
            entry_atr = None

            for i in range(LOOKBACK + 1, len(df)):

                # ================= ENTRY =================
                if not in_position:

                    if MOMENTUM:
                        if df["Close"].iloc[i - 1] > upper.iloc[i - 1]:
                            signals.iloc[i, signals.columns.get_loc(ticker)] = 1
                            in_position = True
                            entry_price = df["Open"].iloc[i]
                            entry_atr = atr.iloc[i - 1]

                    else:
                        if df["Close"].iloc[i - 1] < lower.iloc[i - 1]:
                            signals.iloc[i, signals.columns.get_loc(ticker)] = 1
                            in_position = True
                            entry_price = df["Open"].iloc[i]
                            entry_atr = atr.iloc[i - 1]

                # ================= EXIT =================
                else:
                    sl_price = entry_price - SL_ATR_MULT * entry_atr
                    tp_price = entry_price + TP_SL_MULT * SL_ATR_MULT * entry_atr

                    if df["Low"].iloc[i - 1] <= sl_price:
                        signals.iloc[i, signals.columns.get_loc(ticker)] = -1
                        in_position = False

                    elif df["High"].iloc[i - 1] >= tp_price:
                        signals.iloc[i, signals.columns.get_loc(ticker)] = -1
                        in_position = False

        return signals

    def process_data(self, data: pd.DataFrame):
        self.signals_df = self.generate_signals_multi_ticker(data)
        return self.signals_df, None

    def get_signals(self, trading_state: dict) -> pd.Series:
        ts = trading_state["current_timestamp"]
        return self.signals_df.loc[ts]


In [ ]:
# trades = portfolio.trades.records_readable

# trade_log = trades[trades["Column"] == stock][
#     [
#         "Entry Timestamp",
#         "Exit Timestamp",
#         "Size",
#         "Avg Entry Price",
#         "Avg Exit Price",
#         "PnL",
#         "Return"
#     ]
# ].reset_index(drop=True)

# trade_log


In [ ]:
def signals_to_sticky_weights(signals: pd.DataFrame) -> pd.DataFrame:
    """
    Convert entry/exit signals into sticky portfolio weights.

    Contract:
    - signal[t] is EXECUTED at OPEN[t]
    - +1 = enter
    - -1 = exit
    - 0  = hold

    NEW BEHAVIOR:
    - On HOLD days, weight is NaN (do nothing)
    - On ENTRY day, weight is a positive number
    - On EXIT day, weight is 0.0
    """

    weights = pd.DataFrame(
        np.nan,                     # 👈 CHANGED: default NaN instead of 0.0
        index=signals.index,
        columns=signals.columns
    )

    current_weights = pd.Series(0.0, index=signals.columns)
    free_pool = 1.0

    for date in signals.index:
        day_signal = signals.loc[date]

        # ================= SELL FIRST =================
        sell_stocks = day_signal[day_signal == -1].index

        for stock in sell_stocks:
            if current_weights[stock] > 0:
                free_pool += current_weights[stock]
                current_weights[stock] = 0.0

        # ================= SLOT CALC =================
        current_holdings = (current_weights > 0).sum()
        available_slots = max(0, MAX_STOCKS - current_holdings)

        # ================= BUY =================
        buy_stocks = day_signal[day_signal == 1].index.tolist()

        if available_slots > 0 and free_pool > 0 and len(buy_stocks) > 0:
            # deterministic selection
            selected = buy_stocks[:available_slots]
            allocation = free_pool / len(selected)

            for stock in selected:
                current_weights[stock] = allocation

            free_pool = 0.0

        # ================= SAVE STATE =================
        # 🔥 KEY CHANGE:
        # - ENTRY day  → emit weight
        # - EXIT day   → emit 0.0
        # - HOLD day   → emit NaN

        day_weights = pd.Series(np.nan, index=signals.columns)

        # Entry days
        entry_stocks = day_signal[day_signal == 1].index
        day_weights[entry_stocks] = current_weights[entry_stocks]

        # Exit days
        exit_stocks = day_signal[day_signal == -1].index
        day_weights[exit_stocks] = 0.0

        # HOLD days intentionally left as NaN 👈 THIS IS THE POINT

        weights.loc[date] = day_weights

    return weights


In [ ]:
import pandas as pd
import numpy as np
import tqdm

class Backtester:
    def __init__(self, data: pd.DataFrame, initial_value: float, start_date, strategy):
        self.data = data
        self.initialvalue = initial_value
        self.startdate = start_date
        self.strategy = strategy
        
        # Pre-extract prices to NumPy for speed
        self.open_prices_raw = data.xs("Open", level=1, axis=1)
        self.close_prices_raw = data.xs("Close", level=1, axis=1)
        
        self.open_prices = self.open_prices_raw.values
        self.close_prices = self.close_prices_raw.values
        self.timestamps = data.index
        self.tickers = data.columns.get_level_values(0).unique()
        
        # State variables
        self.portfolio_value = initial_value
        self.cash = initial_value
        self.investment = 0.0
        self.current_index = 0
        
        # Use NumPy for internal tracking, Convert to DF only at the end
        self.pos_array = np.zeros(len(self.tickers), dtype=int)
        self.all_pos_history = np.zeros((len(data), len(self.tickers)), dtype=int)
        self.all_signals = pd.DataFrame(index=self.timestamps, columns=self.tickers)

    def run(self):
        # Initial setup
        self.strategy.process_data(self.data)
        
        # 1. Pre-calculate ALL signals (if strategy allows) 
        # Note: If strategy depends on current 'cash' in real-time, 
        # this must stay inside the loop. Assuming it needs data history:
        for i in tqdm.tqdm(range(1, len(self.data))):
            ts = self.timestamps[i]
            
            # Minimize dictionary creation overhead
            tradingState = {
                "current_data_slice": self.data.iloc[:i], # This is still slow, see note below
                "current_timestamp": ts,
                "positions": pd.Series(self.pos_array, index=self.tickers),
                "investment": self.investment,
                "cash": self.cash,
            }
            
            self.all_signals.loc[ts] = self.strategy.get_signals(tradingState)

        # 2. Pre-calculate weights for the whole period at once
        # This replaces the row-by-row weights_df.loc[ts]
        weights_matrix = signals_to_sticky_weights(self.all_signals.fillna(0)).values
        
        # 3. Fast Execution Loop
        for i in range(1, len(self.data)):
            self.current_index = i
            
            # Current prices (NumPy is much faster than xs/iloc)
            curr_open = self.open_prices[i]
            curr_close = self.close_prices[i]
            
            # Update portfolio value based on overnight movement
            self.portfolio_value = np.sum(self.pos_array * curr_open) + self.cash
            
            # Weight update logic
            row_weights = weights_matrix[i]
            update_mask = ~np.isnan(row_weights)
            
            if np.any(update_mask):
                # Calculate new positions (Vectorized NumPy)
                target_w = row_weights[update_mask]
                target_p = curr_open[update_mask]
                
                # handle division by zero/NaN safely
                with np.errstate(divide='ignore', invalid='ignore'):
                    shares = np.floor((target_w * self.portfolio_value) / target_p)
                    shares = np.nan_to_num(shares, nan=0.0, posinf=0.0, neginf=0.0).astype(int)
                
                self.pos_array[update_mask] = shares
            
            # Recalculate Cash and Investment (PnL)
            self.cash = self.portfolio_value - np.sum(self.pos_array * curr_open)
            self.investment = np.sum(self.pos_array * curr_close)
            self.portfolio_value = self.investment + self.cash
            
            # Save history
            self.all_pos_history[i] = self.pos_array

        # Convert back to DataFrame for vectorbt compatibility
        self.all_positions = pd.DataFrame(
            self.all_pos_history, 
            index=self.timestamps, 
            columns=self.tickers
        )

    def vectorbt_run(self):
        # (This remains largely the same as it's already vectorized)
        filtered_positions = self.all_positions[self.all_positions.index >= self.startdate]
        idx = filtered_positions.index
        cols = filtered_positions.columns

        portfolio = vbt.Portfolio.from_orders(
            price=self.open_prices_raw.loc[idx, cols],
            close=self.close_prices_raw.loc[idx, cols],
            size=filtered_positions.diff().fillna(0).astype(int).mask(lambda x: x == 0),
            init_cash=self.initialvalue,
            freq="1D",
            cash_sharing=True,
            call_seq="auto",
            log=True,
        )
        return portfolio

In [ ]:
params = {
    "LOOKBACK": 20,
    "ATR_PERIOD": 14,
    "SL_ATR_MULT": 5,
    "RR_RATIO": 1.0,
    "MOMENTUM": True,
    "MAX_STOCKS": 20
}


In [ ]:
LOOKBACK     = params["LOOKBACK"]
ATR_PERIOD   = params["ATR_PERIOD"]
SL_ATR_MULT  = params["SL_ATR_MULT"]
TP_SL_MULT   = params["RR_RATIO"]
MOMENTUM     = params["MOMENTUM"]
MAX_STOCKS   = params["MAX_STOCKS"]


In [ ]:
#1️⃣ Precompute indicators once
indicators = precompute_indicators(data)

In [ ]:


# 2️⃣ Create strategy with cached indicators
strategy = Strategy(indicators)

# 3️⃣ Inject strategy into backtester
bt = Backtester(
    data=data,
    initial_value=INITIAL_CAPITAL,
    start_date=data.index[0],
    strategy=strategy
)

# 4️⃣ Run
bt.run()
portfolio = bt.vectorbt_run()


100%|██████████| 1728/1728 [00:00<00:00, 5369.74it/s]


In [ ]:
portfolio.stats()

Start                                2017-01-02 00:00:00
End                                  2023-12-29 00:00:00
Period                                1729 days 00:00:00
Start Value                                    1000000.0
End Value                                 3673418.235805
Total Return [%]                              267.341824
Benchmark Return [%]                          426.536104
Max Gross Exposure [%]                        110.027096
Total Fees Paid                                      0.0
Max Drawdown [%]                               61.709436
Max Drawdown Duration                  735 days 00:00:00
Total Trades                                         644
Total Closed Trades                                  622
Total Open Trades                                     22
Open Trade PnL                             -51358.367043
Win Rate [%]                                   61.414791
Best Trade [%]                                110.000003
Worst Trade [%]                

In [ ]:
stock = "ANANTRAJ.NS"


In [ ]:
close_price = bt.data.xs(
    "Close", level=1, axis=1
)[stock]

open_price = bt.data.xs(
    "Open", level=1, axis=1
)[stock]


In [ ]:
pos = bt.all_positions[stock]

changes = pos.diff().fillna(0)

entry_dates = changes[changes > 0].index
exit_dates  = changes[changes < 0].index


In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

# --- Price line ---
fig.add_trace(
    go.Scatter(
        x=close_price.index,
        y=close_price.values,
        mode="lines",
        name="Close Price",
        hovertemplate=
        "Date: %{x|%d %b %Y}<br>"
        "Price: %{y:.2f}<extra></extra>"
    )
)

# --- Entry markers ---
fig.add_trace(
    go.Scatter(
        x=entry_dates,
        y=close_price.loc[entry_dates],
        mode="markers",
        marker=dict(symbol="triangle-up", size=12, color="green"),
        name="Entry",
        hovertemplate=
        "<b>ENTRY</b><br>"
        "Date: %{x|%d %b %Y}<br>"
        "Price: %{y:.2f}<extra></extra>"
    )
)

# --- Exit markers ---
fig.add_trace(
    go.Scatter(
        x=exit_dates,
        y=close_price.loc[exit_dates],
        mode="markers",
        marker=dict(symbol="triangle-down", size=12, color="red"),
        name="Exit",
        hovertemplate=
        "<b>EXIT</b><br>"
        "Date: %{x|%d %b %Y}<br>"
        "Price: %{y:.2f}<extra></extra>"
    )
)

fig.update_layout(
    title=f"{stock} — Price with Entry & Exit Dates",
    xaxis_title="Date",
    yaxis_title="Price",
    hovermode="x unified",
    xaxis=dict(
        tickformat="%d %b %Y"   # 👈 THIS is what shows month names on axis
    )
)

fig.show()


In [ ]:
equity = portfolio.value()

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=equity.index,
        y=equity.values,
        mode="lines",
        name="Portfolio Equity"
    )
)

fig.update_layout(
    title="Portfolio Equity Curve",
    xaxis_title="Date",
    yaxis_title="Equity Value",
    hovermode="x unified"
)

fig.show()


In [ ]:
shares = bt.all_positions[stock]

stock_value = shares * close_price

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=stock_value.index,
        y=stock_value.values,
        mode="lines",
        name=f"{stock} Position Value"
    )
)

fig.update_layout(
    title=f"{stock} — Position Value Over Time",
    xaxis_title="Date",
    yaxis_title="Value",
    hovermode="x unified"
)

fig.show()


In [ ]:
trades = portfolio.trades.records_readable

trade_log = trades[trades["Column"] == stock][
    [
        "Entry Timestamp",
        "Exit Timestamp",
        "Size",
        "Avg Entry Price",
        "Avg Exit Price",
        "PnL",
        "Return"
    ]
].reset_index(drop=True)

trade_log


,Entry Timestamp,Exit Timestamp,Size,Avg Entry Price,Avg Exit Price,PnL,Return
0,2020-01-13,2020-02-26,1688.0,16.617243,12.831035,-6391.119362,-0.227848
1,2020-06-08,2020-06-11,3310.0,7.908966,9.949311,6753.542585,0.257979
2,2020-08-25,2020-09-22,2136.0,15.986208,11.274483,-10064.245102,-0.294737
3,2021-01-12,2021-01-18,1537.0,29.000000,34.599998,8607.197655,0.193103
4,2021-05-24,2021-09-28,2925.0,61.000000,74.650002,39926.254463,0.223771
5,2021-12-09,2022-04-20,3402.0,78.449997,62.299999,-54942.292213,-0.205864
6,2022-07-06,2022-07-26,918.0,57.500000,77.699997,18543.597198,0.351304
7,2022-08-26,2022-10-06,1179.0,82.000000,102.300003,23933.703598,0.247561
8,2023-04-18,2023-06-13,849.0,142.000000,169.000000,22923.000000,0.190141


In [ ]:
def audit_trade_lifecycle(backtester, ticker):
    import pandas as pd
    
    # 1. Setup data
    signals = backtester.all_signals[ticker]
    positions = backtester.all_positions[ticker]
    prices = backtester.data.xs("Open", level=1, axis=1)[ticker]
    full_index = backtester.data.index
    
    # 2. Find first entry and the very next exit
    entries = signals[signals == 1].index
    if len(entries) == 0:
        print(f"No entry found for {ticker}")
        return
    
    entry_date = entries[0]
    entry_idx = full_index.get_loc(entry_date)
    
    # Find the first exit (-1) AFTER this entry
    exits = signals[(signals == -1) & (signals.index > entry_date)].index
    if len(exits) == 0:
        print(f"Stock {ticker} entered on {entry_date.date()} but has not exited yet.")
        exit_idx = len(full_index) - 1 # Use last available date if not exited
    else:
        exit_date = exits[0]
        exit_idx = full_index.get_loc(exit_date)

    # 3. Define the specific 8 dates you want
    # Entry, E+1, E+2, E+3 ... X-3, X-2, X-1, Exit
    target_indices = [
        entry_idx, entry_idx + 1, entry_idx + 2, entry_idx + 3,
        exit_idx - 3, exit_idx - 2, exit_idx - 1, exit_idx
    ]
    
    # Clean indices: Remove duplicates (if trade is short) and stay within bounds
    target_indices = sorted(list(set([i for i in target_indices if 0 <= i < len(full_index)])))
    audit_dates = full_index[target_indices]

    # 4. Print Log
    print(f"--- Trade Lifecycle Audit: {ticker} ---")
    print(f"Entry: {full_index[entry_idx].date()} | Exit: {full_index[exit_idx].date()}")
    print("-" * 85)
    print(f"{'Date':<12} | {'Signal':<7} | {'Weight':<10} | {'Price (Open)':<12} | {'Shares':<8} | {'Status'}")
    print("-" * 85)
    
    # Pre-calculate sticky weights for the whole period to save time
    # (Using your existing signals_to_sticky_weights function)
    weights_df = signals_to_sticky_weights(backtester.all_signals)

    for date in audit_dates:
        sig = signals.loc[date]
        w = weights_df.loc[date, ticker]
        pos = positions.loc[date]
        px = prices.loc[date]
        
        # Logic display
        if date == entry_date:
            status = "ENTRY"
        elif date == full_index[exit_idx]:
            status = "EXIT"
        elif pd.isna(w):
            status = "STICKY HOLD"
        else:
            status = "HOLD"

        print(f"{str(date.date()):<12} | {sig:<7.0f} | {str(round(w,4)) if pd.notna(w) else 'NaN':<10} | {px:<12.2f} | {pos:<8.0f} | {status}")

# Run the audit
audit_trade_lifecycle(bt, "DBREALTY.NS")

--- Trade Lifecycle Audit: DBREALTY.NS ---
Entry: 2017-02-07 | Exit: 2017-04-18
-------------------------------------------------------------------------------------
Date         | Signal  | Weight     | Price (Open) | Shares   | Status
-------------------------------------------------------------------------------------
2017-02-07   | 1       | 0.0096     | 44.30        | 230      | ENTRY
2017-02-08   | 0       | NaN        | 43.25        | 230      | STICKY HOLD
2017-02-09   | 0       | NaN        | 43.30        | 230      | STICKY HOLD
2017-02-10   | 0       | NaN        | 43.50        | 230      | STICKY HOLD
2017-04-12   | 0       | NaN        | 45.15        | 230      | STICKY HOLD
2017-04-13   | 0       | NaN        | 45.40        | 230      | STICKY HOLD
2017-04-17   | 0       | NaN        | 45.85        | 230      | STICKY HOLD
2017-04-18   | -1      | 0.0        | 52.90        | 0        | EXIT
